In [1]:
import sys
import os
import numpy as  np
from itertools import combinations
import networkx as nx
import matplotlib.pyplot as plt


# Get the root directory (QAOA_mtrix_simulation)File ~\OneDrive - Nottingham Trent University\Desktop\Projects\QAOA_simulation_LV\QAOA_mtrix_simulation\main\Base\__init__.py:32, in choose_simulator(name, mixer_type, **kwargs)

repo_root = r"C:\new_fpga\updated_project\QAOAHP"
sys.path.insert(0, repo_root)
#sys.path.insert(0, r'C:\Users\N1259534\OneDrive - Nottingham Trent University\Desktop\Projects\QAOA_simulation_LV\QAOA_mtrix_simulation')
sys.executable


'c:\\Users\\N1259534\\AppData\\Local\\anaconda3\\envs\\QAOA_matrix\\python.exe'

### Test QAOA core component

In [ ]:
# Simple 2-node graph
G_simple = nx.Graph()
G_simple.add_edge((0, 1), (2,4), weight=1.0)

print(G_simple.edges(data=True))   


In [ ]:
# Simple 2-node graph
np.random.seed(10)
G_simple = nx.Graph()
for i in range(4):
    for j in range(i+1, 4):
        G_simple.add_edge(i, j, weight=1.0)

from main.Base.maxcut import get_maxcut_terms
# Get MaxCut terms
terms = get_maxcut_terms(G_simple)
print(f"Number of terms: {len(terms)}")
print("Terms (coefficient, qubits):")
for i , (coeff, qubits) in enumerate(terms):
    print(f" {i}: coeff= {coeff}, qubits={qubits}")

# Test with your current graph
print( f"\n\n graph G: {G_simple.number_of_nodes()} nodes, {G_simple.number_of_edges()} edge")

terms_g = get_maxcut_terms(G_simple)
print(f" generated {len(terms_g)} terms")
nx.draw(G_simple, with_labels=True)

In [ ]:
from main.Base import choose_simulator
import numpy as np

# Simple problem
N = 3

terms_simple = [(1.0, (0, 2)),(1, (1, 2))] 
#terms = [(np.random.normal(), spin_pair) for spin_pair in combinations(range(N), r=2)]


print(f"Terms is : {terms_simple}")
print("\n")

simmulator = choose_simulator('auto')

sim = simmulator(N, terms = terms_simple)

#get cost diogonal 
cost_diag = sim.get_cost_diagonal()
print(f" The cost diogonal is : {cost_diag}")
print("\n")

for i, cost in enumerate(cost_diag):
    binary = format(i, f'0{N}b')
    print(f"  |{binary}⟩: {cost:8.4f}")

assert len(cost_diag) == 2**N, f" cost diagonal should have {2**N} elements"
print(f"\n✓ Cost diagonal has correct size: {len(cost_diag)}")

In [ ]:
# TEST 2.2: Test with your actual graph G
from main.QAOA_objective import get_qaoa_objective
from main.Base import choose_simulator
import numpy as np

obj = get_qaoa_objective(G = G_simple, simulator= "python")



In [ ]:
# Test single evaluation (p=2)
theta0 = np.array([0.1, 0.2, 0.3, 0.4])
print(f"\nTest theta: {theta0}")

cost_value = obj(theta0)
print(f"Cost Value: {cost_value:.8f}")

# Test multiple evaluations (for optimization)
print("\nTesting 5 random parameter sets:")
for i in range(5):
    theta_rand = np.random.rand(4)
    cost = obj(theta_rand)
    print(f"  Iteration {i+1}: theta={theta_rand}, cost={cost:.6f}")

print("\n✓ Objective function stable over multiple calls")

In [ ]:
# TEST 2.3: Parameter handling
from main import parameter_utils

# Test theta parameterization (default)
theta = np.array([0.1, 0.2, 0.3, 0.4])  # p=2
print(f" te theta is {theta}")

gamma, beta = parameter_utils.convert_to_gamma_beta(theta, parameterization='theta')

print(f"Converted_gamma = {gamma}")
print(f"Converted_beta = {beta}")

theta_reconstructed = np.hstack([gamma, beta])
print(f"Reconstructed theta: {theta_reconstructed}")

if np.allclose(theta, theta_reconstructed):
    print("\n✓ Parameter conversion is consistent")
else:
    print(" is not same")


### Test Integeration 

In [ ]:
import numba.cuda

# Simple graph
G_test = nx.Graph()
G_test.add_edge(0, 1, weight=1.0)
G_test.add_edge(1, 2, weight=1.0)

theta = np.array([0.5, 0.3]) # p =1

# create opjective function
obj = get_qaoa_objective(G = G_test, simulator= "auto")
#single cost value
cost_value = obj(theta)
print(f"objective value of cpu result is : {cost_value:.10f}")

# gpu (if available)
if numba.cuda.is_available():
    obj_gpu = get_qaoa_objective(G = G_test, simulator = "gpu")
    cost_value_gpu = obj_gpu(theta)
    print(f"gpu result objective value: {cost_value:.10f}")

    if np.isclose(cost_value, cost_value_gpu):
        print("\n✓ GPU and CPU results are consistent")
    else:
        print("\n✗ GPU and CPU results differ")
else:
    print("\n GPU is not available ")


In [ ]:
# test optimisation loop 
from scipy.optimize import minimize
from main.QAOA_objective import get_qaoa_objective
P = 2
theta_new = np.random.rand(2*P)
print(f" new theta is {theta_new}")
print(f"the initial cost value is {obj(theta_new):.6f}")

#iteration
iteration_count = [0]
def callback(xk):
    iteration_count[0] +=1
    if iteration_count[0] % 5 ==0:
        print(f"  Iteration {iteration_count[0]}: cost={obj(xk):.6f}")

# optimise
result = minimize(obj, theta_new, method= 'COBYLA', callback= callback,
                   options = {'maxiter': 50})

print(f"final theta {result.x}")
print(f"final cost value is {result.fun:.6f}")
print(f"success: {result.success}")
print(f"totall iteration: {iteration_count[0]}")

if result.fun < obj(theta_new):
    print("\n✓ Optimization improved the cost value")
else:
    print("\n✗ Optimization did not improve the cost value")


### Data validation  (FPGA ready)



In [2]:
from main.Base.Simulators.FPGA import Fpga_sim
from main.Base.maxcut import get_maxcut_terms
from main.Base import choose_simulator


N = 6
#G_fpga = nx.Graph()
#for i in range(N):
#        for j in range(i+1, N):
#            G_fpga.add_edge(i, j, weight=1.0)
G_fpga = nx.random_regular_graph(d=3, n=N, seed=10)

# Optional visualization
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(G_fpga)
#nx.draw(G_fpga, pos, with_labels=True, node_color="lightblue")
#plt.show()

<Figure size 500x500 with 0 Axes>

In [3]:
from main.QAOA_objective import get_qaoa_objective
terms = get_maxcut_terms(G_fpga)
fpga_config = { "port": "COM3", "baudrate": 115200, "max_qubits": 14}
a = get_qaoa_objective(N, G_fpga, simulator='FPGA', fpga_config=fpga_config) 
theta = np.random.rand(4)
theta
#cost_value = a(theta)


array([0.56245915, 0.15603629, 0.24207748, 0.68921028])

In [4]:
from main import parameter_utils
theta = [0.1, 0.2, 0.3, 0.4]
gamma, beta = parameter_utils.convert_to_gamma_beta(theta, parameterization='theta')
cosb , sinb = parameter_utils.generate_mixer_sincos_fpga( beta, p=2)
cosb, sinb


(array([0.29552021, 0.38941834]), array([0.95533649, 0.92106099]))

In [6]:
print(f"cosb: {cosb[0]}")
scaled = parameter_utils.convert_float_to_fixed(cosb[0], P=64, frac=61)
scaled

cosb: 0.29552020666133955


681423202611435904

In [2]:
P = 64
N = 61

SCALE = 1 << N
MIN_VAL = -(1 << (P - 1))
MAX_VAL = (1 << (P - 1)) - 1
MASK_64 = (1 << P) - 1


def float_to_fixed(a: float) -> int:
    """
    Convert float to signed Q3.61 fixed-point integer.
    Saturates if value is outside representable range.
    """
    scaled = int(round(float(a) * SCALE))

    if scaled < MIN_VAL:
        return MIN_VAL
    if scaled > MAX_VAL:
        return MAX_VAL

    return scaled

In [ ]:
# get terms and simulator 
terms = get_maxcut_terms(G_fpga)
print(f"term is {terms}" )

sim = choose_simulator(name='python')(n_qubits=N, terms=terms)
cost_diagonal = sim.get_cost_diagonal()

for i, cost in enumerate(cost_diagonal):
    print(f"  [{i}]: {cost:.10f}  (type: {type(cost).__name__})")

# Check data types
assert all(isinstance(c, (float, np.floating)) for c in cost_diagonal), \
    "All costs should be floats"
print("✓ All costs are float type")


# Check size
assert len(cost_diagonal) == 2**N, f"Should have {2**N} costs"
print(f"✓ Correct size: {len(cost_diagonal)}")

# Initial state
initial_state = np.ones(2**N, dtype=np.complex128) / np.sqrt(2**N)
print(f"\nInitial state:")
for i, amp in enumerate(initial_state):
    print(f"  [{i}]: {amp.real:.6f} + {amp.imag:.6f}j")


# Parameters
gamma = np.array([0.5])
beta = np.array([0.3])
print(f"\nParameters:")
print(f"  gamma: {gamma} (type: {gamma.dtype})")
print(f"  beta:  {beta} (type: {beta.dtype})")
print(f"  cos(β): {np.cos(beta[0]):.10f}")
print(f"  sin(β): {np.sin(beta[0]):.10f}")


### Test FPGA part

**Data need to feed fpga**

1- Cost Diogonal: a array of energy vale for each computational basic state

    type: float64
    size: 2^N
    Target place: BRAM[2] (BRAM_COST_FUNC)

2- Initial state vector (BRAM[0] and BRAM[1])

    type: complex128 (real and imaginary)
    size: 2^N complex numbers
    Target place:  Real part: BRAM[0] (BRAM_STATE_REAL)
                   Imaginary: BRAM[1] (BRAM_STATE_IMAG)

3- QAOA Parameters (BRAM[5])

    Gamma and cos(β) sin(β) - 3xP (3 values per layer)
    Format: float64
    for 




